# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Building the Tumor Screening Dashboard

### Scenario

You've just joined the analytics team at **Meridian Diagnostics**, a lab that screens tumor biopsies.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
np.random.seed(0)
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()
df["diagnosis"] = np.where(df["target"] == 0, "M", "B")
df = df.drop(columns=["target"], errors="ignore")
feature_cols = [c for c in df.columns if c not in ["id", "diagnosis"]]
X_raw = df[feature_cols].values
y = df["diagnosis"].values
print(f"Loaded {df.shape[0]} patients, {len(feature_cols)} measured dimensions.")


In [ ]:
p = len(feature_cols)
n_pairs = comb(p, 2)
n_cells = 10 ** p
print(f"p = {p} features")
print(f"Unique pairwise relationships: {n_pairs}")
print(f"Grid cells needed (10 bins/dim): {n_cells:.2e}")


In [ ]:
corr = df[feature_cols].corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_cols))); ax.set_xticklabels(feature_cols, rotation=90, fontsize=6)
ax.set_yticks(range(len(feature_cols))); ax.set_yticklabels(feature_cols, fontsize=6)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlogram: 30 tumor measurement features")
plt.tight_layout(); plt.show()
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
high_corr_pairs = int((corr.where(mask).abs() > 0.9).sum().sum())
print("Highly correlated pairs (|r| > 0.9):", high_corr_pairs)


In [ ]:
def pca_from_scratch(X, k):
    X_scaled = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
    cov = np.cov(X_scaled, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    Z = X_scaled @ eigvecs[:, :k]
    return Z, eigvals, eigvecs
Z, eigvals, eigvecs = pca_from_scratch(X_raw, k=2)
fig, ax = plt.subplots(figsize=(6, 5))
for label, color in [("B", "#3b6ea5"), ("M", "#b5432e")]:
    mask = y == label
    ax.scatter(Z[mask, 0], Z[mask, 1], s=15, alpha=0.6, color=color, label=label)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title("Your from-scratch PCA"); ax.legend(); plt.show()


In [ ]:
explained_ratio = eigvals / eigvals.sum()
cumulative = np.cumsum(explained_ratio)
k_90 = int(np.argmax(cumulative >= 0.90) + 1)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(range(1, len(cumulative) + 1), cumulative * 100, "o-", color="#3b6ea5")
ax.axhline(90, color="#b5432e", ls="--")
ax.set_xlabel("k"); ax.set_ylabel("Cumulative % variance"); ax.set_title(f"k = {k_90} needed for >= 90% variance"); plt.show()


In [ ]:
X_scaled = StandardScaler().fit_transform(X_raw)
pca_emb = PCA(n_components=2, random_state=0).fit_transform(X_scaled)
tsne_emb = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X_scaled)
umap_emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(X_scaled)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, emb, name in zip(axes, [pca_emb, tsne_emb, umap_emb], ["PCA", "t-SNE", "UMAP"]):
    for label, color in [("B", "#3b6ea5"), ("M", "#b5432e")]:
        mask = y == label
        ax.scatter(emb[mask, 0], emb[mask, 1], s=12, alpha=0.6, color=color, label=label)
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()
